# Readmission Analysis

This notebook examines how recorded prior service use and information from the representative inpatient encounter relate to Any Readmission. The intended prediction point is near discharge.

The analysis uses the full prepared cohort, including patients assigned to Test. The cohort results describe historical records. Document any influence they have on later feature, rule, threshold, or model choices. Those choices are evaluated on the development data under the modeling protocol.

## 1. Cohort and Outcome

Read the prepared release and its target mapping. The positive class `readmitted` combines `<30` and `>30`. The negative class `not_readmitted` comes from `NO`, which indicates no readmission recorded within the source's observation coverage. Each patient contributes one eligible encounter selected by the smallest eligible numeric encounter ID.

In [ ]:
import sys
from pathlib import Path

from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    directory
    for directory in (working_directory, *working_directory.parents)
    if (directory / "src" / "paths.py").is_file() and (directory / "requirements.txt").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.analysis.data import load_analysis_inputs

analysis = load_analysis_inputs(project_root=project_root)

In [ ]:
print(f"Prepared cohort: {len(analysis.target):,} patients and {analysis.predictors.shape[1]} predictors.")
display(analysis.target.value_counts(sort=False).rename("patients").to_frame())

## 2. Statistical Conventions

Let N be the cohort size, Y the total number of recorded readmissions, n the number of patients in a group, and y its recorded readmissions.

| Measure | Calculation |
| --- | --- |
| Coverage | n / N |
| Within-group readmission rate | y / n |
| Share of the cohort's recorded readmissions | y / Y |
| Rule support | y / N |
| Rule confidence | y / n |
| Rule lift | (y / n) / (Y / N) |

A denominator of zero makes the corresponding measure undefined. Tables flag groups below 30 patients or below 5 patients in either outcome class. These thresholds guide interpretation.

## 3. Q1. Prior Service Use

Compare patients with and without recorded outpatient, emergency, or inpatient visits in the previous year. Use the same groups to count how many readmissions a fixed rule based on any recorded prior visit captures or misses.

## 4. Q2. Current Encounter Characteristics

Compare readmission rates by length of stay, number of diagnoses, and first diagnosis group within each prior service group. Interpret encounter fields in the context of the planned prediction point near discharge. Their availability at that point requires verification.

## 5. Q3. Laboratory and Treatment Records

Measure laboratory result coverage and readmission rates by recording pattern. Preserve the source result levels and treatment statuses. The literal laboratory value `None` means no result is recorded. A result-level rate uses patients in that level of the corresponding test.

## 6. Q4. Combined Conditions

Compare each specified pair of conditions with its individual conditions. Report coverage, support, confidence, and lift with the underlying counts. Use these comparisons as references for the later Weka evaluation.